In [70]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [71]:
df_enforcement    = pd.read_csv("cleaned/enforcement_actions.csv")
df_narratives     = pd.read_csv("cleaned/ltc_narratives.csv")
df_facility_types = pd.read_csv("cleaned/lookup_facility_types.csv")

In [94]:
df_enforcement['PENALTY_CATEGORY'].value_counts()

PENALTY_CATEGORY
Patient Care                                                                     8296
Patient Rights                                                                   3096
Abuse/Facility Not Self Reported                                                 1560
Other                                                                             973
AE: Stage 3 or 4 ulcer acquired after admission                                   693
DHPPD: 2 - 11 Days                                                                660
Administration                                                                    567
Breach to person/entity outside facility/hc system                                532
Problem Transfer                                                                  475
Medication                                                                        474
Physical Environment                                                              397
AE: Retention of a foreign object in 

In [89]:
df_enforcement.columns

Index(['FACID', 'FACILITY_NAME', 'LTC', 'FAC_TYPE_CODE', 'FAC_FDR',
       'DISTRICT_OFFICE', 'PENALTY_ISSUE_DATE', 'PENALTY_NUMBER',
       'DISPOSITION', 'PENALTY_TYPE', 'PENALTY_DETAIL', 'PENALTY_CATEGORY',
       'PENALTY_CATEGORY_OTHER', 'VIOLATION_FROM_DATE', 'VIOLATION_TO_DATE',
       'APPEALED', 'APPEAL_DUE_DATE', 'APPEAL_RECEIVED_DATE',
       'CLASS_ASSESSED_INITIAL', 'CLASS_ASSESSED_FINAL',
       'TOTAL_AMOUNT_INITIAL', 'TOTAL_AMOUNT_DUE_FINAL',
       'TOTAL_PENALTY_OFFSET_AMOUNT', 'TOTAL_COLLECTED_AMOUNT',
       'TOTAL_BALANCE_DUE', 'ACLAIMS_VISIT_NUMBER', 'EVENTID', 'DEATH_RELATED',
       'INTAKEID_ALL', 'PRIORITY_ALL', 'SFY', 'HIGHEST_PRIORITY',
       'COMPLAINT_COUNT'],
      dtype='str')

In [ ]:
df_enforcement['FAC_FDR'].value_counts()

I. Load Datasets

In [ ]:
df_enforcement = pd.read_excel("raw/enforcement_actions.xlsx")
df_facility_types = pd.read_excel("raw/lookup_facility_types.xlsx")
df_narratives = pd.read_csv("raw/ltc_narratives.csv", encoding="latin-1")

II. Auditing Column names for all tables to ensure column names do not have to be stripped or have trailing whitespaces. <br><br>
Ex. 'VARIABLE has extra spaces'


In [ ]:
print(df_enforcement.columns)
print(df_facility_types.columns)
print(df_narratives.columns)

In [ ]:
df_enforcement.columns = df_enforcement.columns.str.strip()
df_facility_types.columns = df_facility_types.columns.str.strip()
df_narratives.columns = df_narratives.columns.str.strip()

II. Auditing df_facility_types

In [ ]:
df_facility_types.columns

a. Ensure 'VARIABLE' column records are properly entered and explicitly map to categories in the Data Dictionary. Documentation states that there are 7 categories that 'VARIABLE' can fall under.

In [ ]:
df_facility_types['VARIABLE'].value_counts()

b. Misc 'Variable' records have no significant data, and can consequently be dropped safely

In [ ]:
df_facility_types[
    df_facility_types['VARIABLE'].isin(['VARIABLE', 'VARIABLE  '])]

In [ ]:
df_facility_types = df_facility_types[
    ~df_facility_types['VARIABLE'].isin(['VARIABLE', 'VARIABLE  '])
].reset_index(drop=True)

c. Quick inspect/Sanity Check to make sure these map properly to documentation specifications <br><br>
Found one Null Value in Variable "Penalty" for Description, but this is fine based on specifications. <br><br> *Please note I used head() for notebook conciseness but entire table is useful to inspect for business rule contradictions

In [ ]:
pd.set_option('display.max_rows', None)
df_facility_types[['VARIABLE','DESCRIPTION']].drop_duplicates().sort_values(['VARIABLE','DESCRIPTION']).head()

In [ ]:
df_facility_types[df_facility_types['VARIABLE']=='penalty_category'].head()

d. Making sure codes are correctly entered 

In [ ]:
print(df_enforcement['FAC_TYPE_CODE'].value_counts())


III. Auditing df_narratives

a. Clean-up punctuation in names

In [ ]:
df_narratives['FACILITY_NAME'] = df_narratives['FACILITY_NAME'].str.title()

In [ ]:
# value counts for all categorical columns at once
cat_cols = df_narratives.select_dtypes(include=['object']).columns

for col in cat_cols:
    print(f"COLUMN: {col}")
    print(f"Nulls: {df_narratives[col].isna().sum()}")
    print(df_narratives[col].value_counts(dropna=False).head(10))

Will not drop values missing EVENT_ID, provide lots of other insight 

In [ ]:
df_narratives[df_narratives['EVENTID'].isna()].head()

a. Applying TF-IDF to 'narratives' to get column with top 3 words, additionally stripping original full text 

IV. Auditing df_enforcement

In [ ]:
df_enforcement.columns

In [ ]:
# value counts for all categorical columns at once
cat_cols = df_enforcement.select_dtypes(include=['object']).columns

for col in cat_cols:
    print(f"\n{'='*40}")
    print(f"COLUMN: {col}")
    print(f"Nulls: {df_enforcement[col].isna().sum()}")
    print(df_enforcement[col].value_counts(dropna=False))

a. Penalty Category should map to values in df_facility_types where variable== penalty_category, must standardize so they match (when possible)

In [ ]:

official_categories = df_facility_types[
    df_facility_types['VARIABLE'] == 'penalty_category'
][['VALUE']]

print(official_categories)
print(f"\nOfficial categories: {len(official_categories)}")

actual_categories = set(df_enforcement['PENALTY_CATEGORY'].dropna().unique())
official_values = set(official_categories['VALUE'].unique())

unmatched = actual_categories - official_values
print(f"\nIn enforcement but not in lookup ({len(unmatched)}):")
print(unmatched)

missing = official_values - actual_categories
print(f"\nIn lookup but not in enforcement ({len(missing)}):")
print(missing)

In [ ]:
official = set(
    df_facility_types[
        df_facility_types['VARIABLE'] == 'penalty_category'
    ]['VALUE'].str.strip().unique()
)

# Print official values
print("OFFICIAL LOOKUP VALUES:")
for v in sorted(official):
    print(f"  '{v}'")

In [ ]:
penalty_category_map = {
    # AE codes — standardized to lookup format
    'AE04 - Retention of a foreign object in a patient': 'AE: Retention of a foreign object in a patient',
    'AE17 - Stage 3 or 4 ulcer acquired after admission': 'AE: Stage 3 or 4 ulcer acquired after admission',
    'AE22 - Death associated with a fall': 'AE: Death associated with a fall',
    'AE11 - Suicide/attempted suicide': 'AE: Suicide/attempted suicide',
    'AE12 - Medication error': 'AE: Medication error',
    'AE26  - Sexual assault on a patient': 'AE: Sexual assault on a patient',
    'AE27 - Death/injury from a physical assault': 'AE: Death/injury from a physical assault',
    'AE28 - Adverse event or series of adverse events': 'AE: Adverse event or series of adverse events',
    'AE01 - Surgery performed on a wrong body part': 'AE: Surgery performed on a wrong body part',
    'AE02 - Wrong patient surgery': 'AE: Wrong patient surgery',
    'AE03 - Wrong surgical procedure performed on a patient': 'AE: Wrong surgical procedure performed on a patient',
    'AE05 - Death during or up to 24 hours after surgery': 'AE: Death during or up to 24 hours after surgery',
    'AE06  -Use of contaminated drug, device, or biologic': 'AE: Use of contaminated drug, device, or biologic',
    'AE07 - Use of device other than as intended': 'AE: Use of device other than as intended',
    'AE08 - Death/disability due to intravascular air embolism': 'AE: Death/disability due to intravascular air embolism',
    'AE10 - Death/disability due to disappearance': 'AE: Death/disability due to disappearance',
    'AE13 - Hemolytic reaction': 'AE: Hemolytic reaction',
    'AE14 - Maternal death/disab due to labor/del/post': 'AE: Maternal death/disab due to labor/del/post',
    'AE15 - Death/disability directly related to hypoglycemia': 'AE: Death/disability directly related to hypoglycemia',
    'AE19 - Death/disability due to electric shock': 'AE: Death/disability due to electric shock',
    'AE20 - Line contaminated or use for wrong gas': 'AE: Line contaminated or use for wrong gas',
    'AE21 - Death/disability due to a burn': 'AE: Death/disability due to a burn',
    'AE23 - Death/disab assoc with use of restraints/bedrails': 'AE: Death/disab assoc with use of restraints/bedrails',
    'AE24 - Care by impersonating licensed provider': 'AE: Care by impersonating licensed provider',
    'AE25 - Abduction of a patient of any age': 'AE: Abduction of a patient of any age',

    # Abbreviation fixes
    'Financial Occurrence/Fac Not Self Reported': 'Financial Occurrence/Facility Not Self Reported',
    'Breach of IT system theft/loss of edevice/med rec': 'Breach of IT system theft/loss of device/med records',
    'Deliberate breach of PHI by health care worker': 'Deliberate breach of PHI (protected health information) by health care worker',

    # Severity flags
    'Other Immediate Jeopardy (not an AE)': 'Non-AE AP Immediate Jeopardy',
    'Other Non-Immediate Jeopardy (not an AE)': 'Non-AE AP Non-Immediate Jeopardy',
}

# Apply mapping
df_enforcement['PENALTY_CATEGORY'] = (
    df_enforcement['PENALTY_CATEGORY']
    .str.strip()
    .replace(penalty_category_map)
)

# Validate against lookup — note lowercase 'penalty_category'
official_values = set(
    df_facility_types[
        df_facility_types['VARIABLE'] == 'penalty_category'
    ]['VALUE'].str.strip().unique()
)

actual_values = set(df_enforcement['PENALTY_CATEGORY'].dropna().str.strip().unique())

matched = actual_values & official_values
unmatched = actual_values - official_values

print(f"Matched to official lookup: {len(matched)}")
print(f"Not in lookup (retained as-is): {len(unmatched)}")
print(f"\nRetained categories not in lookup:")
for v in sorted(unmatched):
    print(f"  {v}")

In [ ]:
subset=df_facility_types[df_facility_types['VARIABLE']=='penalty_category']
subset['VALUE'].value_counts()

b. Decomposing PRIORITY_ALL into counts of complaints and the highest severity of complaint

In [ ]:
# 1. highest priority per citation
def get_highest_priority(val):
    if pd.isna(val):
        return None
    priorities = [p.strip() for p in str(val).split(',')]
    order = ['A', 'B', 'C', 'D', 'E']
    for p in order:
        if p in priorities:
            return p
    return None

# 2. count of complaints that preceded this citation
def count_complaints(val):
    if pd.isna(val):
        return 0
    return len(str(val).split(','))

df_enforcement['HIGHEST_PRIORITY'] = df_enforcement['PRIORITY_ALL'].apply(get_highest_priority)
df_enforcement['COMPLAINT_COUNT'] = df_enforcement['PRIORITY_ALL'].apply(count_complaints)

c. Cleaned and stripped narrative column and extracted first 3 words using TFIDF

In [ ]:
os.makedirs("cleaned", exist_ok=True)

def clean_text(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'F\d{3,4}', '', text)          
    text = re.sub(r'T\d{2}', '', text)             
    text = re.sub(r'DIV\d+', '', text)             
    text = re.sub(r'CH\d+', '', text)             
    text = re.sub(r'ART\d+', '', text)             
    text = re.sub(r'\([^)]*\)', '', text)         
    text = re.sub(r'\b\d+\b', '', text)           
    text = re.sub(r'[^\w\s]', ' ', text)           
    text = re.sub(r'\n+', ' ', text)              
    text = re.sub(r'\s+', ' ', text)              
    return text.strip().lower()

print("Cleaning enforcement actions...")
df_enforcement.columns = df_enforcement.columns.str.strip().str.upper()
df_enforcement['PENALTY_NUMBER']    = df_enforcement['PENALTY_NUMBER'].astype(str).str.strip()
df_enforcement['FAC_TYPE_CODE']     = df_enforcement['FAC_TYPE_CODE'].astype(str).str.strip().str.upper()
df_enforcement['FACID']             = df_enforcement['FACID'].astype(str).str.strip()
df_enforcement['PENALTY_ISSUE_DATE']= pd.to_datetime(df_enforcement['PENALTY_ISSUE_DATE'], errors='coerce')
df_enforcement['VIOLATION_FROM_DATE']= pd.to_datetime(df_enforcement['VIOLATION_FROM_DATE'], errors='coerce')
df_enforcement['DEATH_RELATED']     = df_enforcement['DEATH_RELATED'].fillna('N')


print("Cleaning narratives...")
df_narratives.columns = df_narratives.columns.str.strip().str.upper()
df_narratives['PENALTY_NUMBER'] = df_narratives['PENALTY_NUMBER'].astype(str).str.strip()
df_narratives['FACID']          = df_narratives['FACID'].astype(str).str.strip()
df_narratives['NARRATIVE_CLEAN'] = df_narratives['NARRATIVE'].apply(clean_text)


print("Extracting TF-IDF keywords...")
custom_stop_words = list(ENGLISH_STOP_WORDS) + [
    'resident', 'facility', 'patient', 'stated', 'according',
    'don', 'staff', 'nursing', 'hospital', 'following',
    'included', 'indicated', 'required', 'titled', 'dated',
    'review', 'record', 'noted', 'note', 'shall', 'use',
    'used', 'did', 'day', 'days', 'time', 'right', 'left'
]

tfidf = TfidfVectorizer(
    max_features=100,
    stop_words=custom_stop_words,
    ngram_range=(1, 2),
    token_pattern=r'[a-zA-Z]{3,}'
)

matrix = tfidf.fit_transform(df_narratives['NARRATIVE_CLEAN'].fillna(''))
terms  = tfidf.get_feature_names_out()

def get_top_keywords(row, terms, n=3):
    top_indices = np.argsort(row)[-n:][::-1]
    return ', '.join([terms[i] for i in top_indices])

df_narratives['TOP_KEYWORDS'] = [
    get_top_keywords(matrix[i].toarray()[0], terms)
    for i in range(matrix.shape[0])
]
df_narratives = df_narratives.drop(columns=['NARRATIVE'])


df_facility_types.columns = df_facility_types.columns.str.strip().str.upper()
df_facility_types['VALUE'] = df_facility_types['VALUE'].astype(str).str.strip().str.upper()
df_facility_types = df_facility_types.dropna(subset=['VALUE'])


df_enforcement.to_csv("cleaned/enforcement_actions.csv", index=False)
df_narratives.to_csv("cleaned/ltc_narratives.csv", index=False)
df_facility_types.to_csv("cleaned/lookup_facility_types.csv", index=False)

print(f"\nDone!")
print(f"  Enforcement actions: {df_enforcement.shape}")
print(f"  Narratives:          {df_narratives.shape}")
print(f"  Lookup:              {df_facility_types.shape}")
print(f"\nCleaned files saved to cleaned/")

In [ ]:
df_enforcement['CLASS_ASSESSED_FINAL'] = np.where(
    df_enforcement['APPEALED'] == 'No',
    df_enforcement['CLASS_ASSESSED_INITIAL'],  
    df_enforcement['CLASS_ASSESSED_FINAL']  
)

In [ ]:
df_enforcement['CLASS_ASSESSED_FINAL'].isna

In [ ]:
df_enforcement['FAC_TYPE_CODE'].value_counts().sort_index()

In [ ]:
print(df_facility_types['VARIABLE'].unique())

In [ ]:
final_map = {
    'B Trebled'           : 'B TREBLED',
    'A Trebled'           : 'A TREBLED',
    'B First'             : 'B FIRST',
    'Dismissed by Court'  : 'Dismissed by court',
    'CHOW- SETTLEMENT'    : 'CHOW SETTLEMENT',
    'APPEAL WITHDRAWN BY' : 'APPEAL WITHDRAWN BY FACILITY',
    'R/D=<$1000'          : 'R/D = <$1000',
    'FTR RES'             : 'FRTR RES',
    'DEPT WITHDREW CITATI': 'Dismissed by court', 
    'DISMISSED' : 'Dismissed by court',
    'WMF FIRST' : 'WF'
}

df_enforcement['CLASS_ASSESSED_FINAL'] = (
    df_enforcement['CLASS_ASSESSED_FINAL']
    .astype(str)
    .str.strip()
    .str.replace(r'\t', '', regex=True)
    .replace(final_map)
)

24-Hour facility is designated in definition

In [ ]:
fac_mask = df_facility_types['VARIABLE'] == 'FAC_TYPE_CODE'

df_facility_types.loc[fac_mask, 'IS_24HR'] = (
    df_facility_types.loc[fac_mask, 'DEFINITION']
    .str.contains('24-hour', regex=False, na=False)
    .astype(int)
)

df_facility_types.loc[fac_mask, 'IS_HOSPITAL'] = (
    df_facility_types.loc[fac_mask, 'DEFINITION']
    .str.lower()
    .str.contains('governing body|duly constituted', regex=True, na=False)
    .astype(int)
)

df_facility_types.loc[fac_mask, 'IS_LOW_RESOURCE'] = (
    df_facility_types.loc[fac_mask, 'DEFINITION']
    .str.lower()
    .str.contains('nonprofit|non-profit|tax-exempt|community|federally qualified', regex=True, na=False)
    .astype(int)
)

Notable observation: During sanity check count before joining, I found that column names were swapped from what they were shown in documentation

In [ ]:
# 1. fact_enforcement_actions ↔ fact_ltc_narratives on PENALTY_NUMBER
enf_penalty = set(df_enforcement['PENALTY_NUMBER'].dropna().astype(str).str.strip().unique())
nar_penalty  = set(df_narratives['PENALTY_NUMBER'].dropna().astype(str).str.strip().unique())

print("=== PENALTY_NUMBER join ===")
print(f"Enforcement:       {len(enf_penalty)}")
print(f"Narratives:        {len(nar_penalty)}")
print(f"Matching:          {len(enf_penalty & nar_penalty)}")
print(f"Narratives only:   {len(nar_penalty - enf_penalty)}")
print(f"Enforcement only:  {len(enf_penalty - nar_penalty)}")

# 2. fact_enforcement_actions ↔ dim_fac_code on FAC_TYPE_CODE / VALUE
enf_fac_type  = set(df_enforcement['FAC_TYPE_CODE'].dropna().astype(str).str.strip().unique())
dim_fac_value = set(df_facility_types[df_facility_types['VARIABLE'] == 'FAC_TYPE_CODE']['VALUE'].dropna().astype(str).str.strip().unique())

print("\n=== FAC_TYPE_CODE join ===")
print(f"Enforcement:       {len(enf_fac_type)}")
print(f"Dim lookup:        {len(dim_fac_value)}")
print(f"Matching:          {len(enf_fac_type & dim_fac_value)}")
print(f"Enforcement only:  {len(enf_fac_type - dim_fac_value)}")
print(f"Unmatched codes:   {enf_fac_type - dim_fac_value}")

# 3. fact_enforcement_actions ↔ dim_district_office on DISTRICT_OFFICE / VALUE
enf_class = set(df_enforcement['CLASS_ASSESSED_FINAL'].dropna().astype(str).str.strip().unique())
dim_class = set(df_facility_types[
    df_facility_types['VARIABLE'] == 'Class_Assessed_Final'
]['DESCRIPTION'].dropna().astype(str).str.strip().unique())

print("\n=== FAC_TYPE_CODE join ===")
print(f"\nEnforcement:      {len(enf_class)}")
print(f"Dim lookup:       {len(dim_class)}")
print(f"Matching:         {len(enf_class & dim_class)}")
print(f"Enforcement only: {enf_class - dim_class}")
print(f"Lookup only:      {dim_class - enf_class}")

In [ ]:
df_facility_types[df_facility_types['VARIABLE']=='Class_Assessed_Final']['VALUE']

In [ ]:
df_enforcement['PENALTY_CATEGORY'].value_counts()

In [86]:
df_enforcement['PENALTY_DETAIL'].value_counts()

PENALTY_DETAIL
Citation B (HSC 1424)                                       12554
Citation A (HSC 1424)                                        3325
Failure to Report Adverse Events to CDPH (HSC 1280.4)        1185
AP - 3.2 Nursing-Hours-per-Patient-Day AP (HSC 1276.5)       1043
AP - Immediate Jeopardy (HSC 1280.3)                          564
Failure to Report Breach to CDPH (HSC 1280.15(b)(1))          543
Citation AA (HSC 1424)                                        414
AP - Breach (HSC 1280.15)                                     305
AP - Non-Immediate Jeopardy (HSC 1280.3)                      255
Failure to Report Breach to Resident (HSC 1280.15(b)(2))      227
Citation Willful Material Falsification (HSC 1424(f)(1))      104
Citation Willful Material Omission (HSC 1424(f)(1))             8
Citation Retaliation/Discrimination (HSC 1432)                  6
Name: count, dtype: int64

In [82]:
subset=df_facility_types[df_facility_types['VARIABLE']=='Penalty_detail, class_assessed_Initial']
subset['VALUE'].value_counts()

VALUE
Citation AA  (HSC 1424)                                     1
Citation A  (HSC 1424)                                      1
Citation B  (HSC 1424)                                      1
AP - Breach (HSC 1280.15)                                   1
Failure to Report Breach to CDPH (HSC 1280.15(b)(1))        1
Failure to Report Breach to Resident (HSC 1280.15(b)(2))    1
3.2 Nursing-Hours-per-Patient-Day AP (HSC 1276.5)           1
Citation Willful Material Omission (HSC 1424 (f)(1))        1
Citation Willful Material Falsification (HSC 1424(f)(1))    1
Citation Retaliation/Discrimination (HSC 1432)              1
AP - Immediate Jeopardy (HSC 1280.3)                        1
AP - Non-Immediate Jeopardy (HSC 1280.3)                    1
Failure to Report Adverse Events to CDPH (HSC 1280.4)       1
Name: count, dtype: int64

In [106]:
df_enforcement['PENALTY_DETAIL'].value_counts()

PENALTY_DETAIL
Citation B (HSC 1424)                                       12554
Citation A (HSC 1424)                                        3325
Failure to Report Adverse Events to CDPH (HSC 1280.4)        1185
3.2 Nursing-Hours-per-Patient-Day AP (HSC 1276.5)            1043
AP - Immediate Jeopardy (HSC 1280.3)                          564
Failure to Report Breach to CDPH (HSC 1280.15(b)(1))          543
Citation AA (HSC 1424)                                        414
AP - Breach (HSC 1280.15)                                     305
AP - Non-Immediate Jeopardy (HSC 1280.3)                      255
Failure to Report Breach to Resident (HSC 1280.15(b)(2))      227
Citation Willful Material Falsification (HSC 1424(f)(1))      104
Citation Willful Material Omission (HSC 1424 (f)(1))            8
Citation Retaliation/Discrimination (HSC 1432)                  6
Name: count, dtype: int64

In [111]:
df_enforcement['PENALTY_CATEGORY'].value_counts()

PENALTY_CATEGORY
Patient Care                                                                     8296
Patient Rights                                                                   3096
Abuse/Facility Not Self Reported                                                 1560
Other                                                                             973
AE: Stage 3 or 4 ulcer acquired after admission                                   693
DHPPD: 2 - 11 Days                                                                660
Administration                                                                    567
Breach to person/entity outside facility/hc system                                532
Problem Transfer                                                                  475
Medication                                                                        474
Physical Environment                                                              397
AE: Retention of a foreign object in 

In [108]:
df_facility_types[df_facility_types['VARIABLE']=='penalty_category']

,VARIABLE,DESCRIPTION,VALUE,DEFINITION,IS_24HR,IS_HOSPITAL,IS_LOW_RESOURCE
123,penalty_category,NaN,Abuse/Facility Not Self Reported,NaN,NaN,NaN,NaN
124,penalty_category,NaN,Administration,NaN,NaN,NaN,NaN
125,penalty_category,NaN,Dietary,NaN,NaN,NaN,NaN
126,penalty_category,NaN,Family/Council Requirement,NaN,NaN,NaN,NaN
127,penalty_category,NaN,Failure to meet the 3.2 hours of nursing care,NaN,NaN,NaN,NaN
128,penalty_category,NaN,Financial Occurrence/Facility Not Self Reported,NaN,NaN,NaN,NaN
129,penalty_category,NaN,Medication,NaN,NaN,NaN,NaN
130,penalty_category,NaN,Other,NaN,NaN,NaN,NaN
131,penalty_category,NaN,Patient Care,NaN,NaN,NaN,NaN
132,penalty_category,NaN,Patient Record,NaN,NaN,NaN,NaN


In [95]:
subset['VALUE'] = subset['VALUE'].str.strip().str.replace(r'\s+', ' ', regex=True)
df_enforcement['PENALTY_DETAIL'] = df_enforcement['PENALTY_DETAIL'].str.strip().str.replace(r'\s+', ' ', regex=True)

dim_values = set(subset['VALUE'].dropna().unique())
penalty_values = set(df_enforcement['PENALTY_DETAIL'].dropna().unique())

print(f"In both: {len(dim_values & penalty_values)}")

In both: 11


In [99]:
df_enforcement['PENALTY_DETAIL'] = (
    df_enforcement['PENALTY_DETAIL']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .replace({
        'AP - 3.2 Nursing-Hours-per-Patient-Day AP (HSC 1276.5)' : '3.2 Nursing-Hours-per-Patient-Day AP (HSC 1276.5)',
        'Citation Willful Material Omission (HSC 1424(f)(1))'    : 'Citation Willful Material Omission (HSC 1424 (f)(1))',
    })
)

# verify
dim_values     = set(subset['VALUE'].dropna().unique())
penalty_values = set(df_enforcement['PENALTY_DETAIL'].dropna().unique())
print(f"In both: {len(dim_values & penalty_values)}")  # should be 13

In both: 13
